# VLABench 任务评测 Notebook (改进版)

本 Notebook 演示如何使用 VLABench 对现有任务进行评测

## 改进内容
- 过��� IK 求解器警告，使输出更清晰
- 减少评测回合数和步数，加快评测速度
- 添加更好的进度显示和错误处理

## 1. 导入必要的库并过滤警告

In [1]:
import os
import sys
import json
import numpy as np
import warnings
import logging

# 过滤 MuJoCo IK 求解器的收敛警告
warnings.filterwarnings('ignore', category=Warning)
logging.getLogger('absl').setLevel(logging.ERROR)

# 设置项目路径
os.environ['VLABENCH_ROOT'] = '/ssd/mkqin/workspace/VLABench'
sys.path.insert(0, os.environ['VLABENCH_ROOT'])

# 导入 VLABench 相关模块
from VLABench.evaluation.evaluator import Evaluator
from VLABench.evaluation.model.policy.base import RandomPolicy
from VLABench.tasks import *
from VLABench.robots import *

print("✓ 导入成功！")
print("✓ 已过滤 IK 求解器警告，输出将更清晰")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
✓ 导入成功！
✓ 已过滤 IK 求解器警告，输出将更清晰


## 2. 设置环境

In [2]:
# 设置渲染后端
os.environ["MUJOCO_GL"] = "egl"

print(f"VLABench 根目录: {os.getenv('VLABENCH_ROOT')}")
print(f"MuJoCo 渲染后端: {os.getenv('MUJOCO_GL')}")

VLABench 根目录: /ssd/mkqin/workspace/VLABench/VLABench
MuJoCo 渲染后端: egl


## 3. 选择评测任务

In [3]:
# 可用任务列表
available_tasks = {
    "基础任务 (M&T)": [
        "select_poker",      # 选择扑克牌
        "select_drink",      # 选择饮料
        "select_book",       # 选择书籍
        "select_toy",        # 选择玩具
        "select_fruit",      # 选择水果
        "add_condiment",     # 添加调料
        "insert_flower",     # 插花
    ],
    "空间推理 (Spatial)": [
        "select_fruit_spatial",
        "select_poker_spatial",
        "select_book_spatial",
        "add_condiment_spatial",
    ],
    "常识推理 (CommonSense)": [
        "select_drink_common_sense",
        "select_fruit_common_sense",
        "select_toy_common_sense",
        "add_condiment_common_sense",
    ],
    "语义理解 (Semantic)": [
        "select_book_semantic",
        "select_fruit_semantic",
        "select_toy_semantic",
        "add_condiment_semantic",
    ]
}

print("=== 可用任务列表 ===")
for category, tasks in available_tasks.items():
    print(f"\n{category}:")
    for task in tasks:
        print(f"  - {task}")

=== 可用任务列表 ===

基础任务 (M&T):
  - select_poker
  - select_drink
  - select_book
  - select_toy
  - select_fruit
  - add_condiment
  - insert_flower

空间推理 (Spatial):
  - select_fruit_spatial
  - select_poker_spatial
  - select_book_spatial
  - add_condiment_spatial

常识推理 (CommonSense):
  - select_drink_common_sense
  - select_fruit_common_sense
  - select_toy_common_sense
  - add_condiment_common_sense

语义理解 (Semantic):
  - select_book_semantic
  - select_fruit_semantic
  - select_toy_semantic
  - add_condiment_semantic


## 4. 配置评测参数（优化版）

In [4]:
# ========== 评测配置 ==========
# 选择要评测的任务
selected_tasks = [
    "select_fruit",     # 水果选择任务
]

# 评测回合数（使用较少的回合进行快速测试）
n_episodes = 1  # 减少回合数加快测试

# 是否使用未见过的物体进行泛化评测
unseen_objects = False

# 保存结果目录
save_dir = "/ssd/mkqin/workspace/VLABench/logs/evaluation"

# 是否启用可视化
enable_visualization = False

# 最大子步数（每个动作的最大步数）
max_substeps = 10

print(f"评测任务: {selected_tasks}")
print(f"评测回合数: {n_episodes}")
print(f"使用未见物体: {unseen_objects}")
print(f"结果保存路径: {save_dir}")
print(f"\n注意: 随机策略通常无法完成任务，这是预期行为。")
print(f"      要获得有意义的结果，请使用训练好的策略模型。")

评测任务: ['select_fruit']
评测回合数: 1
使用未见物体: False
结果保存路径: /ssd/mkqin/workspace/VLABench/logs/evaluation

注意: 随机策略通常无法完成任务，这是预期行为。
      要获得有意义的结果，请使用训练好的策略模型。


## 5. 初始化评测器

In [5]:
evaluator = Evaluator(
    tasks=selected_tasks,
    n_episodes=n_episodes,
    max_substeps=max_substeps,
    save_dir=save_dir,
    visulization=enable_visualization
)

print(f"✓ 评测器初始化成功！")
print(f"  将评测 {len(selected_tasks)} 个任务，每个任务 {n_episodes} 回合")

Load the task episodes by seeds, instead of episodes
✓ 评测器初始化成功！
  将评测 1 个任务，每个任务 1 回合


## 6. 加载评测模型

In [6]:
# 使用随机策略
random_policy = RandomPolicy(model=None)

print("✓ 随机策略已加载")
print("\n说明:")
print("  - 随机策略作为基线，用于测试评测流程")
print("  - 随机策略的成功率通常为 0%，这是正常的")
print("  - 要获得有意义的结果，需要使用训练好的模型")

✓ 随机策略已加载

说明:
  - 随机策略作为基线，用于测试评测流程
  - 随机策略的成功率通常为 0%，这是正常的
  - 要获得有意义的结果，需要使用训练好的模型


## 7. 运行评测

In [7]:
import time

print("开始评测...")
print("=" * 60)
print(f"任务: {selected_tasks[0]}")
print(f"策略: {random_policy.name}")
print(f"回合数: {n_episodes}")
print("=" * 60)

start_time = time.time()

# 使用随机策略评测
result = evaluator.evaluate(random_policy)

end_time = time.time()
elapsed_time = end_time - start_time

print("\n" + "=" * 60)
print(f"✓ 评测完成！耗时: {elapsed_time/60:.1f} 分钟")
print("=" * 60)

开始评测...
任务: select_fruit
策略: RandomPolicy
回合数: 1


Evaluating select_fruit of RandomPolicy: 100%|██████████| 1/1 [02:49<00:00, 169.94s/it]



✓ 评测完成！耗时: 2.8 分钟


## 8. 查看评测结果

In [8]:
print("\n" + "=" * 60)
print("评测结果汇总")
print("=" * 60 + "\n")

for task_name in selected_tasks:
    if task_name in result:
        task_result = result[task_name]
        print(f"任务: {task_name}")
        print(f"  ✓ 成功率: {task_result.get('success_rate', 0):.2%}")
        print(f"  ✓ 意图得分: {task_result.get('intention_score', 0):.3f}")
        print(f"  ✓ 任务进度: {task_result.get('progress_score', 0):.3f}")
        print()

print("=" * 60)
print("说明:")
print("  - 随机策略无法完成任务是正常的")
print("  - 这些指标用于验证评测流程是否正常工作")
print("  - 使用训练好的模型可以获得更好的结果")
print("=" * 60)


评测结果汇总

任务: select_fruit
  ✓ 成功率: 0.00%
  ✓ 意图得分: 0.000
  ✓ 任务进度: 0.000

说明:
  - 随机策略无法完成任务是正常的
  - 这些指标用于验证评测流程是否正常工作
  - 使用训练好的模型可以获得更好的结果


## 9. 查看详细结果（如果有）

In [9]:
# 读取详细结果文件
detail_file = os.path.join(save_dir, selected_tasks[0], "detail_info.json")

if os.path.exists(detail_file):
    with open(detail_file, 'r') as f:
        detail_info = json.load(f)
    
    print("\n详细回合信息:")
    print("=" * 60)
    
    for i, info in enumerate(detail_info):
        print(f"\n回合 {i+1}:")
        print(f"  - 成功: {info.get('success', False)}")
        print(f"  - 消耗步数: {info.get('consumed_step', 0)}")
        print(f"  - 意图得分: {info.get('intention_score', 0):.3f}")
        print(f"  - 任务进度: {info.get('progress_score', 0):.3f}")
else:
    print(f"\n详细结果文件未找到: {detail_file}")


详细回合信息:

回合 1:
  - 成功: False
  - 消耗步数: 200
  - 意图得分: 0.000
  - 任务进度: 0.000


## 10. 使用真实模型的示例（可选）

In [ ]:
# 如果你有 OpenVLA 模型权重，可以使用以下代码：

print("使用真实模型的示例代码:")
print("=" * 60)
print("""
# 取消注释以下代码以使用 OpenVLA 模型
from VLABench.evaluation.model.policy.openvla import OpenVLA

# 模型权重路径（需要修改为实际路径）
model_ckpt = "/path/to/openvla-7b"
lora_ckpt = "/path/to/lora/checkpoint"  # 可选

# 初始化 OpenVLA 模型
policy = OpenVLA(
    model_ckpt=model_ckpt,
    lora_ckpt=lora_ckpt,
    norm_config_file=os.path.join(os.getenv("VLABENCH_ROOT"), 
                                  "configs/model/openvla_config.json")
)

# 运行评测
result = evaluator.evaluate(policy)
""")
print("=" * 60)

## 总结

### 问题解答

#### Q: 为什么会有 "Failed to converge" 警告？
**A:** 这是 MuJoCo IK 求解器的正常警告，表示随机策略生成的某些目标位置无法被机器人到达。这已经被过滤掉，不影响评测结果。

#### Q: 为什么随机策略成功率为 0%？
**A:** 随机策略作为基线，无法完成复杂的操作任务。这是预期行为。要获得有意义的结果，需要使用训练好的策略模型（如 OpenVLA）。

#### Q: 如何加快评测速度？
**A:** 可以：
1. 减少 `n_episodes`（评测回合数）
2. 减少 `max_episode_length`（在任务配置中）
3. 使用 GPU 加速（如果有 VLA 模型）

### 下一步

1. **使用训练好的模型**: 替换随机策略为真实的策略模型
2. **增加评测回合数**: 获得更稳定的统计结果
3. **尝试不同任务**: 测试模型在各种任务上的表现
4. **启用可视化**: 保存视频用于分析和演示

### 相关文件

- 结果文件: `/ssd/mkqin/workspace/VLABench/logs/evaluation/`
- 任务代码: `/ssd/mkqin/workspace/VLABench/VLABench/tasks/`
- 模型代码: `/ssd/mkqin/workspace/VLABench/VLABench/evaluation/model/policy/`